# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR^2 colorectal cancer dataset using the [`mlcroissant`](https://mlcroissant.org) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Install mlcroissant library if not installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Date Published: {metadata.datePublished}")

## 2. Data Overview

Review available record sets, fields, and their IDs.

Note: The `mlcroissant` library exposes record set `@id`s and field `@id`s, which should be used for referencing data elements.

In [ ]:
# List available record sets and their @id
record_sets = [rec for rec in dataset.record_sets()]
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}, Name: {rs.get('name', '')}")

# Explore fields in each record set
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} ({rs.get('name', '')})")
    fields = rs.get('field', [])
    # If only a dict, wrap as list
    if isinstance(fields, dict):
        fields = [fields]
    for fld in fields:
        print(f"  Field @id: {fld['@id']}, Name: {fld.get('name', '')}, DataType: {fld.get('dataType', '')}")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from the main record set
# Find the clinical record set by name or @id
clinical_rs_id = None
clinical_rs_name = None
for rs in record_sets:
    # Choose the record set with relevant fields (e.g., tabular clinical data)
    if rs.get('name', '').lower().startswith('clinicopathological'):
        clinical_rs_id = rs['@id']
        clinical_rs_name = rs['name']
        break
if not clinical_rs_id:
    # fallback: select first record set or manual id
    clinical_rs_id = record_sets[0]['@id']
    clinical_rs_name = record_sets[0].get('name', '')

# Load records for main record set
records = list(dataset.records(record_set=clinical_rs_id))
df = pd.DataFrame(records)

print(f"Loaded {len(df)} records from record set '{clinical_rs_name}' (@id: {clinical_rs_id}).")
print(f"Columns (@id): {df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations may include removing outliers, transforming data distributions, or grouping data by key attributes.

**Note:** The fields and columns are referenced by their `@id` in accordance with Croissant schema.

In [ ]:
# Choose a numeric field for analysis, e.g., interval between cancers
# Find numeric field @id
numeric_field_id = None
group_field_id = None

# Try to select a field with integer/float data type
for rs in record_sets:
    if rs['@id'] == clinical_rs_id:
        for field in rs.get('field', []):
            dt = field.get('dataType', '').lower()
            if ('integer' in dt or 'float' in dt or 'number' in dt) and 'interval' in field.get('name', '').lower():
                numeric_field_id = field['@id']
            if 'location' in field.get('name', '').lower():
                group_field_id = field['@id']
            # Fallback: select first integer/float field
            if not numeric_field_id and ('integer' in dt or 'float' in dt or 'number' in dt):
                numeric_field_id = field['@id']
            # Fallback: first groupable field (categorical)
            if not group_field_id and dt == 'schema:text':
                group_field_id = field['@id']

if not numeric_field_id:
    # Fallback: pick first numeric column in dataframe
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

if not group_field_id:
    # Fallback: pick first object column
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]):
            group_field_id = col
            break

print(f"Numeric field chosen for EDA (@id): {numeric_field_id}")
print(f"Grouping field chosen for EDA (@id): {group_field_id}")

# Filter records with numeric_field > threshold (e.g., interval > 10)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize numeric_field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by categorical/group field and calculate mean
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    grouped_df.columns = [f"Mean of {numeric_field_id}"]
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

**Note:** All field references are by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(7,4))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=10, kde=True)
plt.title(f"Distribution of {numeric_field_id} (filtered > {threshold})")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Boxplot grouped by group_field
if group_field_id in filtered_df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
    plt.xticks(rotation=45)
    plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

This notebook demonstrates step-by-step data exploration of the FAIR^2 colorectal cancer dataset using [`mlcroissant`](https://mlcroissant.org):

- **Dataset Metadata:** FAIR^2 dataset describes clinicopathological and molecular characteristics of second primary colorectal cancer in survivors.
- **Data Overview:** Record sets, fields, and columns are referenced strictly by their Croissant `@id`.
- **Data Extraction:** Main clinical record set loaded into Pandas DataFrame.
- **EDA:** Numeric and categorical analysis, normalization, and grouping performed by field `@id`.
- **Visualization:** Distributions and relationships visualized for key numeric and group fields.

Further analyses can be performed using this workflow, ensuring reproducibility and compliance with FAIR data practices.
